# YOLO V5 — verified balanced 80/10/10 split, 20-epoch YOLO11s training

This creates a NEW split of the same 2,261 V5 photographs, targeting 1,809 train / 226 validation / 226 test. All 97 previously held-out validation images and 71 test images remain outside training. Duplicate and visually near-duplicate groups stay together. Exact totals may differ slightly to preserve groups.

Seven classes: food, plate, utensil, hinged_container, drink_carton, wrapper, bottle.

The newest reported V5 archive is YOLO_V5_CommandCenter_B187_QC_selected_merged1572.zip. Upload the correct complete V5 ZIP, not worker-assignment archives. The original photos and labels are not modified; only the split changes.

Important: V5 still contains provisional annotations, unresolved QC flags, and potentially incomplete cross-class labeling of the V4 subset. A new split ensures class representation but does not establish human-verified ground truth. Do not claim independent seven-class performance yet.

In Google Colab: Runtime → Change runtime type → T4 GPU, then run the cells in order. No Google Drive required.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pyyaml" "pillow" "imagehash"

## 1. Select a verified complete V5 dataset ZIP

The notebook **ignores leftover V4 ZIPs** in Colab and only accepts a complete seven-class V5 ZIP containing `v5_original_plus_v4/manifest.json` with 2,261 entries (2,093 train / 97 val / 71 test). If no valid V5 ZIP is present, it prompts you to upload one. With several valid V5 ZIPs, it asks which corrected version to use.

**Do not upload a four-class V4-only ZIP**, worker-assignment ZIP or label-patch-only ZIP. The complete V5 dataset was not published as a GitHub repository file. You need your original merged V5 archive.

In [ ]:
from pathlib import Path
from google.colab import files
from collections import Counter, defaultdict
import hashlib, json, zipfile, shutil, math, csv
import yaml

# Only a complete seven-class V5 archive with 2,261 manifest entries is valid.
LATEST_ARCHIVE_NAME = "YOLO_V5_CommandCenter_B187_QC_selected_merged1572.zip"
EXPECTED_LATEST = LATEST_ARCHIVE_NAME
EXPECTED_ROOT = "v5_original_plus_v4"
EXPECTED_SPLITS = Counter({"train": 2093, "val": 97, "test": 71})
CLASSES = ["food", "plate", "utensil", "hinged_container", "drink_carton", "wrapper", "bottle"]
TARGET = {"train": 1809, "val": 226, "test": 226}
DATASET_DIR = Path("/content") / EXPECTED_ROOT
ROOT = DATASET_DIR

def check_v5_zip(path):
    try:
        with zipfile.ZipFile(path) as z:
            names = [n.replace("\\", "/") for n in z.namelist() if not n.endswith("/")]
            required = f"{EXPECTED_ROOT}/manifest.json"
            if required not in names:
                found = ", ".join(sorted({n.split("/")[0] for n in names})[:5]) or "(empty)"
                return False, f"wrong dataset structure: found root(s) {found}; expected {EXPECTED_ROOT}/"
            if any(not n.startswith(EXPECTED_ROOT + "/") or ".." in Path(n).parts
                   or Path(n).is_absolute() for n in names):
                return False, "archive contains unexpected or unsafe paths"
            rows = json.loads(z.read(required))
            if not isinstance(rows, list) or len(rows) != 2261:
                return False, f"manifest needs 2,261 images; found {len(rows) if isinstance(rows, list) else 'non-list data'}"
            actual = Counter(r.get("split") for r in rows)
            if actual != EXPECTED_SPLITS:
                return False, f"unexpected original splits: {dict(actual)}"
            for r in rows:
                if not all(k in r for k in ("image_path", "label_path", "image_sha256", "label_sha256")):
                    return False, "incomplete V5 manifest"
                if f"{EXPECTED_ROOT}/{r['image_path']}" not in names or f"{EXPECTED_ROOT}/{r['label_path']}" not in names:
                    return False, "missing image or label files referenced by manifest"
            return True, "complete V5 archive with correct manifest and splits"
    except (OSError, zipfile.BadZipFile, ValueError, KeyError, TypeError) as exc:
        return False, f"unreadable/incompatible archive: {exc}"

def valid_archives():
    choices = []
    for candidate in sorted(Path("/content").glob("*.zip")):
        good, why = check_v5_zip(candidate)
        print(("✅ " if good else "⛔ ") + candidate.name + ": " + why)
        if good:
            choices.append(candidate)
    return choices

candidates = valid_archives()
if not candidates:
    print("\nNo valid V5 ZIP is currently in /content.")
    print("The V4 ZIP (root yolo_v4_four_class) cannot train the seven-class V5 model.")
    print("Upload a complete V5 ZIP such as:", LATEST_ARCHIVE_NAME)
    uploaded = files.upload()
    del uploaded
    candidates = valid_archives()
    if not candidates:
        raise FileNotFoundError(
            "No COMPLETE 2,261-image V5 archive found. Upload the V5 archive, NOT "
            "a V4 four-class ZIP, a worker assignment ZIP, or a patch ZIP."
        )

if len(candidates) == 1:
    BASE_ZIP = candidates[0]
else:
    print("\nMultiple complete V5 archives found. Choose the latest corrected one:")
    for i, p in enumerate(candidates, start=1):
        print(f"{i}: {p.name}")
    answer = input("Archive number to use: ").strip()
    if not answer.isdigit() or not (1 <= int(answer) <= len(candidates)):
        raise ValueError("Enter a listed archive number and rerun this cell.")
    BASE_ZIP = candidates[int(answer)-1]

archive = BASE_ZIP
assert check_v5_zip(BASE_ZIP)[0], "Selected ZIP is not a valid full V5 archive."
print("\n✅ Selected complete V5 ZIP:", BASE_ZIP.name)
if BASE_ZIP.name != LATEST_ARCHIVE_NAME:
    print("Confirm this ZIP includes your latest corrected annotation labels.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

sha256 = sha256_file

## 2. Extract the V5 archive and verify every label and source image

ZIP CRC and SHA256 checks detect corruption or stale manifest records; they cannot prove every box is semantically correct.

In [ ]:
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
with zipfile.ZipFile(BASE_ZIP) as z:
    assert z.testzip() is None, "ZIP CRC failed"
    for member in z.infolist():
        p = Path(member.filename.replace("\\", "/"))
        assert p.parts and p.parts[0] == DATASET_DIR.name, f"Wrong archive root: {member.filename}"
        assert not p.is_absolute() and ".." not in p.parts, f"Unsafe ZIP member: {member.filename}"
    z.extractall("/content")

manifest = json.loads((DATASET_DIR / "manifest.json").read_text(encoding="utf-8"))
assert len(manifest) == 2261
assert Counter(r["split"] for r in manifest) == {"train": 2093, "val": 97, "test": 71}

image_paths, label_counts = [], []
for r in manifest:
    ip = DATASET_DIR / r["image_path"]
    lp = DATASET_DIR / r["label_path"]
    assert ip.is_file() and lp.is_file(), f"Missing image or label: {r}"
    assert sha256_file(ip) == r["image_sha256"], f"Image checksum mismatch: {ip}"
    assert sha256_file(lp) == r["label_sha256"], f"Label checksum mismatch: {lp}"
    counts = Counter()
    for ln in lp.read_text(encoding="utf-8-sig").splitlines():
        if not ln.strip():
            continue
        parts = ln.split()
        assert len(parts) == 5 and parts[0] in {str(c) for c in range(7)}, f"Invalid label: {lp}"
        cx,cy,w,h = map(float, parts[1:])
        assert all(map(math.isfinite, (cx,cy,w,h)))
        assert 0 < w <= 1 and 0 < h <= 1
        assert -0.00001 <= cx-w/2 <= cx+w/2 <= 1.00001
        assert -0.00001 <= cy-h/2 <= cy+h/2 <= 1.00001
        counts[int(parts[0])] += 1
    image_paths.append(ip)
    label_counts.append(counts)

print("Verified 2,261 image+label pairs and YOLO geometry")
print("Old splits:", dict(Counter(r["split"] for r in manifest)))
total_boxes = sum(label_counts, Counter())
print("All class boxes:", {CLASSES[c]: total_boxes[c] for c in range(7)})

## 3. Build a new group-aware 80/10/10 split

Exact-image matches and conservative perceptual hashes prevent obvious duplicate leakage. Recognized pair/tray/meal identifiers are also grouped if present. The old validation and test images remain held out; the extra validation/test images come only from the old training pool. Rare-class coverage is prioritized.

If related camera-session images are not detected as near-duplicates, additional manual grouping may be needed. Reported evaluation remains exploratory.

In [ ]:
import imagehash, random
from PIL import Image, ImageOps

n = len(manifest)
parent = list(range(n))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a,b):
    a,b = root(a),root(b)
    if a != b:
        parent[max(a,b)] = min(a,b)

# Group byte-identical photographs.
seen = {}
for i,r in enumerate(manifest):
    sig = r["image_sha256"]
    if sig in seen: union(i, seen[sig])
    else: seen[sig] = i

# Honor explicit per-meal/tray/pair group IDs when available in the manifest.
for key in ("pair_id", "tray_pair_id", "meal_id", "group_id"):
    by_id = defaultdict(list)
    for i,r in enumerate(manifest):
        value = r.get(key)
        if value is not None and str(value).strip():
            scope = str(r.get("source_dataset",r.get("source",r.get("dataset",""))))
            by_id[(scope,str(value))].append(i)
    for inds in by_id.values():
        if 1 < len(inds) <= 60:
            for i in inds[1:]: union(inds[0],i)

# Cross-source conservative near-duplicate checks.
ph,dh,rat = [],[],[]
for path in image_paths:
    with Image.open(path) as image:
        im = ImageOps.exif_transpose(image)
        rat.append(im.width / im.height)
        ph.append(int(str(imagehash.phash(im)),16))
        dh.append(int(str(imagehash.dhash(im)),16))
for i in range(n):
    for j in range(i+1,n):
        if abs(rat[i]-rat[j]) / max(rat[i],rat[j]) > 0.025: continue
        if (ph[i]^ph[j]).bit_count() <= 3 and (dh[i]^dh[j]).bit_count() <= 4:
            union(i,j)

members = defaultdict(list)
for i in range(n): members[root(i)].append(i)
group = {}
for gid,inds in members.items():
    box = sum((label_counts[i] for i in inds),Counter())
    held = {manifest[i]["split"] for i in inds}
    fixed = "test" if "test" in held else ("val" if "val" in held else None)
    group[gid] = {"members":inds, "size":len(inds), "boxes":box,
                  "classes":{c for c in range(7) if box[c]>0}, "fixed":fixed}

TARGET = {"train":1809, "val":226, "test":226}
selected = {}
stats = {s:{"size":0,"boxes":Counter()} for s in ("val","test")}
for gid,g in group.items():
    if g["fixed"]:
        s = g["fixed"]
        selected[gid] = s
        stats[s]["size"] += g["size"]
        stats[s]["boxes"].update(g["boxes"])
remaining = set(group) - set(selected)
assert all(stats[s]["size"] <= TARGET[s]+10 for s in ("val","test")), (
    "A near-duplicate cluster makes the requested split unsafe; inspect group sizes."
)

def select(gid,s):
    selected[gid]=s
    remaining.remove(gid)
    stats[s]["size"]+=group[gid]["size"]
    stats[s]["boxes"].update(group[gid]["boxes"])
def groups_for_class(s,c):
    return sum(c in group[gid]["classes"] for gid,split in selected.items() if split==s)

freq = {c:sum(c in g["classes"] for g in group.values()) for c in range(7)}
for c in sorted(range(7), key=lambda x:(freq[x],x)):
    minimum = 2 if freq[c]>=6 else 1
    for s in ("val","test"):
        while groups_for_class(s,c)<minimum:
            fits = [gid for gid in remaining if c in group[gid]["classes"]
                    and stats[s]["size"]+group[gid]["size"]<=TARGET[s]]
            assert fits, f"Cannot hold out enough independent {CLASSES[c]} groups in {s}"
            def coverage_key(gid):
                g = group[gid]
                new_classes=sum(stats[s]["boxes"][k]==0 and k in g["classes"] for k in range(7))
                return (-new_classes,g["size"],-min(g["boxes"][c],5),gid)
            select(min(fits,key=coverage_key),s)

goals={c:total_boxes[c]*226/n for c in range(7)}
def score(gid,s):
    g=group[gid]
    old=stats[s]["boxes"]
    delta=0
    for c in range(7):
        desired=max(1.0,goals[c])
        delta+=((old[c]+g["boxes"][c]-desired)**2-(old[c]-desired)**2)/desired
    return (delta/g["size"],g["size"],gid)

for s in ("val","test"):
    while stats[s]["size"]<TARGET[s]:
        space=TARGET[s]-stats[s]["size"]
        options=[gid for gid in remaining if group[gid]["size"]<=space]
        if not options: break
        select(min(options,key=lambda gid:score(gid,s)),s)
for gid in remaining: selected[gid]="train"

assign={}
for gid,g in group.items():
    for i in g["members"]: assign[i]=selected[gid]
sizes=Counter(assign.values())
boxes={s:Counter() for s in ("train","val","test")}
for i,counts in enumerate(label_counts): boxes[assign[i]].update(counts)

assert len(assign)==n and sum(sizes.values())==n
assert all(assign[i]!="train" for i,r in enumerate(manifest) if r["split"]!="train")
assert all(len({assign[i] for i in g["members"]})==1 for g in group.values())
for s in ("val","test"):
    assert 0.08*n<=sizes[s]<=0.12*n, f"Too far from target: {s} has {sizes[s]} images"
    missing=[CLASSES[c] for c in range(7) if boxes[s][c]==0]
    assert not missing, f"{s} missing annotated classes: {missing}; stop for QC."

print("✅ Duplicate-safe groups:",len(group),"largest:",max(g["size"] for g in group.values()))
print("NEW SPLIT:",{s:{"images":sizes[s],"percent":round(100*sizes[s]/n,2)} for s in ("train","val","test")})
for s in ("train","val","test"):
    print(s,"boxes:",{CLASSES[c]:boxes[s][c] for c in range(7)})
print("WARNING: Class coverage is NOT evidence that labels are exhaustive or correct.")

## 4. Materialize the new dataset and record split provenance

Creates fresh train/val/test folders. Each photograph keeps its original label text, but gets a unique filename. Original images are not altered.

In [ ]:
NEW_DIR = Path("/content/v5_balanced_801010")
if NEW_DIR.exists(): shutil.rmtree(NEW_DIR)
for s in ("train","val","test"):
    (NEW_DIR/"images"/s).mkdir(parents=True)
    (NEW_DIR/"labels"/s).mkdir(parents=True)

with (NEW_DIR/"split_manifest.csv").open("w",newline="",encoding="utf-8") as f:
    w=csv.writer(f)
    w.writerow(["index","original_split","new_split","duplicate_group",
                "original_image","original_label","image_sha256","label_sha256"])
    for i,r in enumerate(manifest):
        s=assign[i]
        stem=f"v5_{i:05d}"
        ext=Path(r["image_path"]).suffix.lower()
        shutil.copy2(DATASET_DIR/r["image_path"], NEW_DIR/"images"/s/(stem+ext))
        shutil.copy2(DATASET_DIR/r["label_path"], NEW_DIR/"labels"/s/(stem+".txt"))
        w.writerow([i,r["split"],s,root(i),r["image_path"],r["label_path"],
                    r["image_sha256"],r["label_sha256"]])

data_file=NEW_DIR/"data.yaml"
data_file.write_text(yaml.safe_dump({
    "path":str(NEW_DIR),"train":"images/train","val":"images/val",
    "test":"images/test","names":{c:n for c,n in enumerate(CLASSES)}
},sort_keys=False),encoding="utf-8")

split_report={
    "archive_name":BASE_ZIP.name,
    "archive_sha256":sha256_file(BASE_ZIP),
    "old_split":{"train":2093,"val":97,"test":71},
    "target_split":TARGET,"actual_split":dict(sizes),
    "class_box_counts":{s:{CLASSES[c]:boxes[s][c] for c in range(7)} for s in ("train","val","test")},
    "duplicate_groups":len(group),
    "previously_held_out_images_used_for_training":0,
    "epochs":20,
    "status":"Exploratory only: annotations not independently gold-standard verified"
}
(NEW_DIR/"split_report.json").write_text(json.dumps(split_report,indent=2)+"\n")
print("✅ Rebalanced data:",data_file)
print(json.dumps(split_report,indent=2))

## 5. Train YOLO11s for 20 epochs

Keeps the V5 pretrained architecture and 800-pixel resolution. Does not resume prior V5 checkpoints.

In [ ]:
import torch
from ultralytics import YOLO
assert torch.cuda.is_available(), "Enable a T4 GPU in Colab Runtime settings."
print("GPU:",torch.cuda.get_device_name(0))
model=YOLO("yolo11s.pt")
model.train(
    data=str(data_file),
    epochs=20, patience=20,
    imgsz=800, batch=4, device=0,
    optimizer="AdamW", lr0=0.0005, weight_decay=0.0005,
    workers=2, seed=57, deterministic=True, amp=True, cos_lr=True,
    degrees=7, translate=0.1, scale=0.25, fliplr=0.5,
    flipud=0, mosaic=0.5, mixup=0, close_mosaic=5,
    plots=True, project="/content/v5_balanced_runs",
    name="V5_80_10_10_20_epochs", exist_ok=False
)
run_dir=Path(model.trainer.save_dir)
best_pt=run_dir/"weights"/"best.pt"
assert best_pt.is_file()
print("✅ Best checkpoint:",best_pt)

## 6. Train exactly 20 epochs (GPU-safe)

Starts from YOLO11s pretrained weights, not the previous V5 checkpoint. Uses 800-pixel images, AdamW, batch 4 on T4, and falls back to batch 2 if CUDA runs out of memory. Saves checkpoints every five epochs. Patience is 20 so normal early stopping cannot interrupt the 20-epoch schedule.

Download best.pt as soon as training finishes; Colab runtime files disappear if the session resets.

In [ ]:
import torch, gc
from pathlib import Path
from ultralytics import YOLO

assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
EPOCHS = 20

def train_once(batch, name):
    net = YOLO("yolo11s.pt")
    net.train(
        data=str(yaml_path),
        epochs=EPOCHS, patience=EPOCHS,
        imgsz=800, batch=batch, device=0,
        optimizer="AdamW", lr0=0.0005, weight_decay=0.0005,
        workers=2, seed=57, deterministic=True, amp=True, cos_lr=True,
        degrees=7, translate=0.1, scale=0.25, fliplr=0.5,
        flipud=0, mosaic=0.5, mixup=0, close_mosaic=5,
        cache=False, save=True, save_period=5, plots=True,
        project="/content/V5_balanced_training",
        name=name, exist_ok=False
    )
    return Path(net.trainer.save_dir)

try:
    run_dir = train_once(4, "YOLO11s_20epochs_batch4")
    final_batch = 4
except (RuntimeError, torch.cuda.OutOfMemoryError) as error:
    if "out of memory" not in str(error).lower() and "cuda oom" not in str(error).lower():
        raise
    print("CUDA OOM at batch 4; retrying a fresh 20-epoch run at batch 2.")
    gc.collect()
    torch.cuda.empty_cache()
    run_dir = train_once(2, "YOLO11s_20epochs_batch2")
    final_batch = 2

best = run_dir / "weights" / "best.pt"
last = run_dir / "weights" / "last.pt"
assert best.is_file() and last.is_file(), "Training checkpoints missing."
print("✅ Finished 20 epochs. Batch:", final_batch, "Best checkpoint:", best)

## 7. Download checkpoint, then evaluate seven-class validation

The next cell immediately exports best.pt before the longer validation step. The final cell computes precision, recall, mAP50, and mAP50–95 for each labeled class, then exports the split manifest and metrics. The test set remains unevaluated.

In [ ]:
from google.colab import files
print("Downloading best.pt checkpoint before validation...")
files.download(str(best))

In [ ]:
output=Path("/content/YOLO_V5_Balanced_20Epochs_Results.zip")
with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED,compresslevel=4) as z:
    for path,name in [
        (best_pt,"weights/best.pt"),
        (NEW_DIR/"split_manifest.csv","split_manifest.csv"),
        (NEW_DIR/"split_report.json","split_report.json"),
        (NEW_DIR/"data.yaml","data.yaml"),
        (report_file,"validation_report.json"),
        (run_dir/"results.csv","training_results.csv"),
        (run_dir/"confusion_matrix.png","confusion_matrix.png"),
        (run_dir/"confusion_matrix_normalized.png","confusion_matrix_normalized.png"),
    ]:
        if path.exists(): z.write(path,arcname=name)
print("✅ Download archive:",output,round(output.stat().st_size/2**20,1),"MiB")
files.download(str(output))